# Модель на CoBaLD

In [6]:
pip install pytorch-crf

In [7]:
pip install pyconll

In [8]:
import numpy as np
import pandas as pd
import pyconll
from collections import Counter
import random
import os

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

from transformers import Trainer, TrainingArguments
from transformers import BertTokenizer
from transformers import BertModel
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

In [ ]:
# Устанавливаем сиды для воспроизводимости
def set_seed(seed_value=12345):
    random.seed(seed_value) # Задаём сид для встроенного генератора случайных чисел
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed_value)
        torch.backends.cudnn.deterministic = True # чтобы операции свёртки и другие слои давали одинаковый результат

set_seed(12345)

In [9]:
class CustomCoNLLDataset(Dataset):
    '''
    Читает файл в формате конлу, собирает предложения (списки токенов) и списки
    семантических меток, строит словари label2id и id2label для преобразования
    меток в числовые индексы
    '''
    def __init__(self, conllu_file, tokenizer, max_length=256, target_column=-1):
        self.data, self.labels = [], set()
        current_sentence, current_labels = [], []

        # Открываем файл
        with open(conllu_file, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'): # Если строка пустая или начинается с # — это либо конец предложения, либо комментарий
                    if not line and current_sentence:
                        self.data.append((current_sentence.copy(), current_labels.copy()))  # Собираем все токены одного предложения и сохраняем
                        current_sentence, current_labels = [], []
                    continue

                # Для непустых строк разбиваем по табуляции
                parts = line.split('\t')
                if parts[0].isdigit() or '-' in parts[0]:
                    word = parts[1]
                    # Целевой столбец с семантикой последний
                    # Если столбцов меньше 11, метка - O
                    sem_class = parts[target_column] if len(parts) > 10 else 'O'
                    current_sentence.append(word)
                    current_labels.append(sem_class)
                    self.labels.add(sem_class)

            # После цикла мог остаться последний набор токенов
            if current_sentence:
                self.data.append((current_sentence, current_labels))

        # Сортируем метки для консистентного маппинга
        self.tokenizer = tokenizer
        self.max_length = max_length
        # Анализируем распределение меток
        # Считаем, сколько раз встречается каждая метка
        flat_labels = [lbl for _, labels in self.data for lbl in labels]
        self.label_counts = Counter(flat_labels)
        print(f"Label distribution: {self.label_counts}")

        # O должен быть на первой позиции, если он есть
        sorted_labels = sorted(self.labels)
        if 'O' in sorted_labels:
            sorted_labels.remove('O')
            sorted_labels = ['O'] + sorted_labels

        # Делаем два словаря: метка -> число и число -> метка
        self.label2id = {l: i for i, l in enumerate(sorted_labels)}
        self.id2label = {i: l for l, i in self.label2id.items()}
        print(f"Labels: {self.label2id}")

    def __len__(self):
        '''Возвращает число предложений в датасете'''
        return len(self.data)

    def __getitem__(self, idx):
        # Достаём n-е предложение и список его семантических меток
        tokens, labels = self.data[idx]
        text = ' '.join(tokens) # Cклеиваем токены в строку

        # Токенизируем текст через берт-токенайзер
        encoding = self.tokenizer(
            text,
            truncation=True, # обрезаем слишком длинные
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt', # получаем тензоры
            return_offsets_mapping=True,
            return_special_tokens_mask=True,
            is_split_into_words=False
        )

        # Подготовка меток
        # По умолчанию все -100, питон их игнорирует в лоссе
        token_labels = torch.ones(self.max_length, dtype=torch.long) * -100

        # Пословная токенизация
        token_to_word_mapping = {}
        word_idx = 0
        # Получаем список токенов из input_ids
        for i, token in enumerate(self.tokenizer.convert_ids_to_tokens(encoding['input_ids'][0])):
            # Пропускаем специальные токены [CLS], [SEP], [PAD]
            if encoding['special_tokens_mask'][0][i] == 1:
                continue

            # Определяем, к какому слову относится токен
            if token.startswith('##'): # Если токен начинается с ##, это продолжение предыдущего слова
                if i > 0 and i-1 in token_to_word_mapping:
                    token_to_word_mapping[i] = token_to_word_mapping[i-1]
            else:
                # Или считаем, что это начало нового слова
                if word_idx < len(labels):
                    token_to_word_mapping[i] = word_idx
                    word_idx += 1

        # Присваиваем метки
        for token_idx, word_idx in token_to_word_mapping.items():
            if word_idx < len(labels):
                token_labels[token_idx] = self.label2id.get(labels[word_idx], 0)

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': token_labels
        }


class SemanticModel(nn.Module):
    """
    Модель для токен-классификации семантики
    Принимает на вход input_ids и attention_mask, выдаёт логи для каждого токена,
    а при передаче labels считает loss
    """
    def __init__(self, model_name, num_labels, dropout_rate=0.2):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name) # Загружаем предобученный берт
        self.dropout = nn.Dropout(dropout_rate) # Добавляем дропаут для регуляризации
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels) # Линейный слой, выводящий num_labels классов на каждый токен

        # Инициализируем веса классификатора для лучшей сходимости
        self.classifier.weight.data.normal_(mean=0.0, std=0.02) # Это поможет модели сходиться чуть быстрее
        self.classifier.bias.data.zero_() # Инициализируем смещения нулями

    def forward(self, input_ids, attention_mask, labels=None):
        # Прогоняем input_ids через берта, получаем скрытые состояния [batch, seq_len, hidden]
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(outputs.last_hidden_state)
        logits = self.classifier(sequence_output)

        loss = None
        if labels is not None:
            # Если передали метки, считаем CrossEntropyLoss по всем токенам
            # Игнорируем метки -100 (падинг и специальные токены)
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fct(logits.view(-1, logits.shape[-1]), labels.view(-1)) # Преобразуем в [batch*seq_len, num_labels] и [batch*seq_len]
        # Возвращаем либо словарь с loss и logits, либо только logits
        return {'loss': loss, 'logits': logits} if loss is not None else {'logits': logits}


def train_and_eval_semantic(conllu_train,
                           conllu_dev,
                           model_name='DeepPavlov/rubert-base-cased',
                           device=None,
                           epochs=15,
                           batch_size=8,
                           max_length=256,
                           lr=5e-5,
                           warmup_ratio=0.1,
                           weight_decay=0.01,
                           gradient_accumulation_steps=2,
                           early_stopping_patience=3):

    device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Токенайзер и датасеты
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    print("Loading datasets...")
    train_ds = CustomCoNLLDataset(conllu_train, tokenizer, max_length=max_length)
    dev_ds = CustomCoNLLDataset(conllu_dev, tokenizer, max_length=max_length)

    # Проверяем, что лейблы совпадают
    if train_ds.label2id != dev_ds.label2id:
        print("WARNING: Label mappings differ between train and dev sets!")
        print(f"Train: {train_ds.label2id}")
        print(f"Dev: {dev_ds.label2id}")
        # Используем маппинг из тренировочного набора для обоих
        dev_ds.label2id = train_ds.label2id
        dev_ds.id2label = train_ds.id2label

    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    dev_dl = DataLoader(dev_ds, batch_size=batch_size*2, shuffle=False)

    # Вычисляем веса классов для борьбы с дисбалансом
    num_classes = len(train_ds.label2id)
    label_counts = Counter()
    for _, labels in train_ds.data:
        label_counts.update(labels)

    total = sum(label_counts.values())
    # Веса обратно пропорциональны частоте класса
    class_weights = torch.ones(num_classes, device=device)
    for label, idx in train_ds.label2id.items():
        if label in label_counts and label_counts[label] > 0:
            class_weights[idx] = total / (num_classes * label_counts[label])

    # Ограничиваем максимальный вес, чтобы избежать числовой нестабильности
    class_weights = torch.clamp(class_weights, 0.1, 10.0)
    print(f"Class weights: {class_weights}")

    # Создаем модель с нуля
    model = SemanticModel(model_name, num_classes, dropout_rate=0.3).to(device)

    # Настраиваем оптимизатор с различными скоростями обучения
    no_decay = ['bias', 'LayerNorm.weight']
    optimizer_grouped_parameters = [
        {
            'params': [p for n, p in model.bert.named_parameters()
                      if not any(nd in n for nd in no_decay)],
            'weight_decay': weight_decay,
            'lr': lr
        },
        {
            'params': [p for n, p in model.bert.named_parameters()
                      if any(nd in n for nd in no_decay)],
            'weight_decay': 0.0,
            'lr': lr
        },
        {
            'params': [p for n, p in model.classifier.named_parameters()],
            'weight_decay': weight_decay,
            'lr': lr * 10  # Более высокая скорость для классификатора
        }
    ]

    optimizer = AdamW(optimizer_grouped_parameters)

    # Настраиваем расписание обучения
    # Настраиваем линейный scheduler с warmup
    total_steps = epochs * len(train_dl) // gradient_accumulation_steps
    warmup_steps = int(warmup_ratio * total_steps)
    scheduler = get_linear_schedule_with_warmup(optimizer,
                                                num_warmup_steps=warmup_steps,
                                                num_training_steps=total_steps)

    # Обучение и валидация
    best_f1 = 0.0
    no_improvement_count = 0

    for epoch in range(1, epochs+1):
        model.train()
        train_loss = 0.0
        optimizer.zero_grad()

        for step, batch in enumerate(train_dl):
            # Перемещаем данные на устройство
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # Прямой проход и loss
            outputs = model(input_ids, attention_mask, labels)
            loss = outputs['loss'] / gradient_accumulation_steps
            loss.backward()

            train_loss += loss.item() * gradient_accumulation_steps

            # Обновление весов каждые gradient_accumulation_steps шагов
            if (step + 1) % gradient_accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

        train_loss = train_loss / len(train_dl)
        print(f"[Train] Epoch {epoch}/{epochs}  Loss={train_loss:.4f}")

        model.eval()
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for batch in dev_dl:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].cpu().numpy()

                outputs = model(input_ids, attention_mask)
                logits = outputs['logits']
                preds = torch.argmax(logits, dim=-1).cpu().numpy()

                # Разбираем по примерам, фильтруем паддинг
                for i in range(preds.shape[0]):
                    pred = preds[i][batch['attention_mask'][i] == 1]
                    label = labels[i][batch['attention_mask'][i] == 1]

                    # Фильтруем специальные токены
                    mask = label != -100
                    all_preds.extend(pred[mask])
                    all_labels.extend(label[mask])

        # Рассчитываем метрики
        acc = accuracy_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds, average='macro')

        # Показываем отчет по классам для более глубокого анализа
        if epoch % 5 == 0 or epoch == epochs:
            # Получаем только присутствующие в данных классы
            unique_labels = sorted(set(all_labels + all_preds))
            actual_label_names = [train_ds.id2label[i] for i in unique_labels if i in train_ds.id2label]

            report = classification_report(
                all_labels, all_preds,
                labels=unique_labels,  # Используем только реальные метки
                target_names=actual_label_names,  # Используем только реальные имена
                digits=4
            )
            print(f"Classification Report:\n{report}")

        print(f"[Dev]   Epoch {epoch}/{epochs}  Acc={acc:.4f}  F1={f1:.4f}")

        # Сохраняем лучшую модель
        if f1 > best_f1:
            best_f1 = f1
            torch.save({
                'model_state_dict': model.state_dict(),
                'label2id': train_ds.label2id,
                'id2label': train_ds.id2label,
                'f1': f1,
                'acc': acc
            }, 'best_semantic_model.pt')
            print(f"New best model saved (F1={best_f1:.4f})")
            no_improvement_count = 0
        else:
            no_improvement_count += 1

        # Ранняя остановка, если нет улучшения
        if no_improvement_count >= early_stopping_patience:
            print(f"Early stopping after {epoch} epochs without improvement")
            break

    # Загружаем лучшую модель
    checkpoint = torch.load('best_semantic_model.pt')
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    print(f"Best model F1: {checkpoint['f1']:.4f}, Acc: {checkpoint['acc']:.4f}")
    return model, tokenizer, checkpoint['label2id']


class SarcasmDataset(Dataset):
    """
    Датасет торча для классификации сарказма, обогащённый предвычисленными семантическими фичами
    Каждому примеру соответствует:
      1) текст
      2) метка сарказма
      3) семантический вектор sem_pooled
    """
    def __init__(self, df, tokenizer, max_length=128):
        self.texts = df['text'].tolist()
        self.labels = df['sarcasm'].tolist()
        self.sem_feats = df['sem_pooled'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], # Токенизируем n-ый текст:
            truncation=True, # Обрезаем до max_length,
            padding='max_length', # Добавляем паддинг до max_length
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'sem_feats': torch.tensor(self.sem_feats[idx], dtype=torch.float),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long),
        }


class SarcasmClassifier(nn.Module):
    def __init__(self, bert_model, sem_dim, num_classes):
        super().__init__()
        self.bert = bert_model
        # Линейный слой для проекции sem_feats в размерность скрытого состояния ,берта
        self.sem_proj = nn.Linear(sem_dim, bert_model.config.hidden_size)
        self.dropout = nn.Dropout(0.1)
        # Классификатор, принимает конкатенацию [bert_out; sem_proj] и выдаёт num_classes логитов
        self.classifier = nn.Linear(bert_model.config.hidden_size*2, num_classes)

    def forward(self, input_ids, attention_mask, sem_feats, labels=None):
        # Получаем pooled_output из BERT: [batch_size, hidden_size]
        bert_out = self.bert(input_ids=input_ids, attention_mask=attention_mask).pooler_output
        # Проецируем семантические фичи в hidden_size: [batch_size, hidden_size]
        sem_proj = self.sem_proj(sem_feats)
        # Склеиваем векторы берта и семантики вдоль размерности признаков
        joint = torch.cat([bert_out, sem_proj], dim=1)
        joint = self.dropout(joint)
        # Логиты для каждого класса сарказма
        logits = self.classifier(joint)

        if labels is not None:
            # Вычисляем CrossEntropyLoss, сравнивая логиты и истинные метки
            loss = nn.CrossEntropyLoss()(logits, labels)
            return {'loss': loss, 'logits': logits}
        return {'logits': logits}


if __name__ == "__main__":
    # Устройство
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Пути к данным
    train_conllu = r'/content/train.conllu'
    dev_conllu = r'/content/dev.conllu'

    # Обучение семантической модели с улучшенными параметрами
    sem_model, tokenizer, label2id = train_and_eval_semantic(
        conllu_train=train_conllu,
        conllu_dev=dev_conllu,
        model_name='DeepPavlov/rubert-base-cased',
        device=device,
        epochs=5,
        batch_size=8,
        max_length=256,
        lr=5e-5,
        warmup_ratio=0.1,
        weight_decay=0.01,
        gradient_accumulation_steps=2,
        early_stopping_patience=3
    )

    # Модель готова к использованию
    sem_model.eval()

Using device: cuda
Loading datasets...
Label distribution: Counter({'_': 80540, 'PREPOSITION': 45482, 'BEING': 45479, 'CH_REFERENCE_AND_QUANTIFICATION': 30680, 'ORGANIZATION': 17512, 'TIME': 12358, 'COUNTRY_AS_ADMINISTRATIVE_UNIT': 10350, 'VERBAL_COMMUNICATION': 9703, 'COORDINATING_CONJUNCTIONS': 8507, 'CONJUNCTIONS': 5416, 'CH_OF_CONNECTIONS': 5414, 'DISCOURSIVE_UNITS': 5387, 'MODALITY': 5344, 'INHABITED_LOCALITY': 5091, 'PARTICLES': 4907, 'ENTITY_OR_SITUATION_PRONOUN': 4155, 'MOTION': 3883, 'AUXILIARY_VERBS': 3344, 'CH_DEGREE': 3221, 'ARRANGEMENTS': 3155, 'BE': 3081, 'MONEY': 2972, 'TO_COMMIT': 2769, 'RESULTS_OF_GIVING_INFORMATION_AND_SPEECH_ACTIVITY': 2716, 'TRANSPORT': 2682, 'TO_GIVE': 2646, 'STATE_OF_MIND': 2553, 'TO_TAKE_PLACE': 2321, 'CH_DISPOSITION_AND_MOTION': 2274, 'PHYSICAL_PSYCHIC_CONDITION': 2169, 'POSITION_IN_SPACE': 1938, 'DOCUMENT': 1878, 'INFORMATION': 1829, 'EMOTIONS_AND_THEIR_EXPRESSION': 1818, 'PLACE': 1795, 'CIRCUMSTANCE': 1783, 'LAWS_AND_STANDARDS': 1710, 'EXISTEN

pytorch_model.bin:   0%|          | 0.00/714M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of the model checkpoint at DeepPavlov/rubert-base-cased were not used when initializing BertModel: ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


[Train] Epoch 1/5  Loss=1.3619
[Dev]   Epoch 1/5  Acc=0.9031  F1=0.6220
New best model saved (F1=0.6220)
[Train] Epoch 2/5  Loss=0.3644
[Dev]   Epoch 2/5  Acc=0.9547  F1=0.8171
New best model saved (F1=0.8171)
[Train] Epoch 3/5  Loss=0.1847
[Dev]   Epoch 3/5  Acc=0.9758  F1=0.9097
New best model saved (F1=0.9097)
[Train] Epoch 4/5  Loss=0.0994
[Dev]   Epoch 4/5  Acc=0.9865  F1=0.9521
New best model saved (F1=0.9521)
[Train] Epoch 5/5  Loss=0.0553


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Classification Report:
                                                   precision    recall  f1-score   support

                                 ABILITY_OF_BEING     1.0000    0.9773    0.9885        44
                                        ACCESSORY     0.8889    1.0000    0.9412         8
                                              ACT     0.9833    1.0000    0.9916        59
                                         ACTIVITY     1.0000    0.9623    0.9808        53
                             ACTIVITY_BY_INTEREST     1.0000    1.0000    1.0000         1
              ADMINISTRATIVE_AND_TERRITORIAL_UNIT     1.0000    1.0000    1.0000         4
                            ADMINISTRATIVE_REGION     0.9887    0.9804    0.9845       357
                                        ADVENTURE     1.0000    1.0000    1.0000         3
                                        AGGREGATE     1.0000    0.9844    0.9921        64
                      AGGREGATE_OF_LIVING_OBJECTS     1.0000    1.

In [10]:
# Загружаем сырые данные
df = pd.read_csv('/content/dataset_all_data (2).csv')
df['sarcasm'] = df['sarcasm'].astype(int)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Пред–вычисление векторов sem_pooled для каждого текста в df
sem_model.eval().to(device)
sem_pooled_list = []
# Проходим по всем строкам с текстом
for text in df['text'].tolist():
    enc = tokenizer(
        text,
        truncation=True,
        padding='max_length',
        max_length=128,
        return_tensors='pt'
    ).to(device)
    with torch.no_grad(): # Отключаем градиенты, нам не нужно обновлять веса модели
        logits = sem_model(
            input_ids=enc['input_ids'],
            attention_mask=enc['attention_mask']
        )['logits'] # Прогоняем через семантическую модель, получаем логиты [1, seq_len, sem_dim]

    # Убираем размер batch 1, усредняем по всем токенам, чтобы получить фиксированный вектор
    pooled = logits.squeeze(0).mean(dim=0).cpu().numpy()  # [sem_dim]
    sem_pooled_list.append(pooled)


# Вписываем полученные эмбеддинги в новый столбец DataFrame
df['sem_pooled'] = sem_pooled_list

# DataLoader
ds = SarcasmDataset(df, tokenizer, max_length=128)
dl = DataLoader(ds, batch_size=16, shuffle=True)

# Создаём и обучаем SarcasmClassifier
bert = sem_model.bert
sarcasm_model = SarcasmClassifier(
    bert_model=bert,
    sem_dim=df['sem_pooled'].iloc[0].shape[0],
    num_classes=2
).to(device)

# Замораживаем все веса бертовской части классификатора
# Обучаем только дополнительную голову (sem_proj + classifier)
for param in sarcasm_model.bert.parameters():
    param.requires_grad = False

#optimizer = AdamW(sarcasm_model.parameters(), lr=2e-5)

optimizer = AdamW(
    filter(lambda p: p.requires_grad, sarcasm_model.parameters()),
    lr=2e-5
)

sarcasm_model.train()
for epoch in range(3):
    total_loss = 0.0
    for batch in dl:
        optimizer.zero_grad()
        out = sarcasm_model(
            input_ids=batch['input_ids'].to(device),
            attention_mask=batch['attention_mask'].to(device),
            sem_feats=batch['sem_feats'].to(device),
            labels=batch['labels'].to(device)
        )
        out['loss'].backward()
        optimizer.step()
        total_loss += out['loss'].item()
    print(f"Sarcasm Epoch {epoch+1}, Loss={total_loss/len(dl):.4f}")

# Сохраняем модель
torch.save(sarcasm_model.state_dict(), 'sarcasm_model.pt')
print("Training complete.")

Sarcasm Epoch 1, Loss=0.4599
Sarcasm Epoch 2, Loss=0.4397
Sarcasm Epoch 3, Loss=0.4323
Training complete.


In [11]:
# Генерация sem_pooled для всех текстов — выполняем один раз
sem_pooled_list = []
sem_model.eval()
for text in df['text'].tolist():
    enc = tokenizer(
        text,
        truncation=True,
        padding='max_length',
        max_length=128,
        return_tensors='pt'
    ).to(device)
    with torch.no_grad():
        logits = sem_model(
            input_ids=enc['input_ids'],
            attention_mask=enc['attention_mask']
        )['logits']  # [1, seq_len, num_labels]

    # Усредняем по длине последовательности
    pooled = logits.squeeze(0).mean(dim=0).cpu().numpy()  # [num_labels]
    sem_pooled_list.append(pooled)

# Сохраняем в df и на диск
df['sem_pooled'] = sem_pooled_list
df.to_pickle('enriched_sarcasm_pooled.pkl')
print("Pre-computed semantic features for", len(df), "texts.")

Pre-computed semantic features for 9144 texts.


In [12]:
abc = pd.read_pickle(r'/content/enriched_sarcasm_pooled.pkl')

In [13]:
abc

,text,gender,age,sarcasm,sem_pooled
0,Быстро работает паблик. Его уже вылечили и он ...,2,34,0,"[-0.9770956, -2.1125183, -1.5974305, -1.427997..."
1,"это был я ,если вы о комментариях.А вы прям са...",2,43,1,"[-0.9942447, -2.5271513, -1.0031344, -2.899618..."
2,"Георгий, вам начислено 50 + к вашей карме. Ско...",2,43,1,"[-2.0389981, -2.8862143, -2.0044491, -2.307334..."
3,"Вы рачьё поганое , тот, кто проигнорировал ком...",2,29,0,"[-0.2181628, -1.1815004, -0.8001303, -1.238070..."
4,вы плять гоните пластик заваривать?по возможно...,2,43,0,"[-1.6035901, 0.23780707, -1.23159, -1.7988427,..."
...,...,...,...,...,...
9139,Спасибо золотце))))) и ты мне тем же))))) и те...,1,17,0,"[-1.613815, -1.6248398, -2.3401177, -1.6518227..."
9140,"Игорь, вы просто супер поработали!!!!С вами бы...",1,24,0,"[-1.4302878, -1.8017485, -1.4836771, -3.391699..."
9141,Мне (лифтинг ампулы под глаза 2 шт) и куда опл...,1,35,0,"[-2.0699382, -2.5613785, -1.9792006, -2.316885..."
9142,В Рязани научились коптить грудинку? Не-ве-рю!...,2,40,0,"[-2.053646, -0.6720343, -1.7879078, -1.7938467..."


In [30]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import spacy
import os

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [21]:
y = abc['sarcasm']
X = abc.drop('sarcasm', axis=1)

In [24]:
def feature_engineering(choice_transformer, choice_ngrams):
    # Обработка текстовых данных: либо TF-IDF, либо мешок слов
    text_features = 'text'
    if choice_transformer == 'tfidf':
        text_transformer = TfidfVectorizer(
            ngram_range=choice_ngrams,
            tokenizer=word_tokenize,
            stop_words='english'
        )
    else:
        text_transformer = CountVectorizer(
            ngram_range=choice_ngrams,
            tokenizer=word_tokenize,
            stop_words='english'
        )

    # Применяем трансформацию только к текстовому столбцу 'text'
    preprocessor = ColumnTransformer(
        transformers=[
            ("txt", text_transformer, text_features)
        ]
    )
    return preprocessor

In [25]:
def modelfit(model):
    model.fit(Xtrain, ytrain)

    # Предсказания меток
    ypredtest = model.predict(Xtest)
    ypredtrain = model.predict(Xtrain)

    # Предсказания вероятностей (для roc_auc_score)
    yprobtest = model.predict_proba(Xtest)
    yprobtrain = model.predict_proba(Xtrain)

    print('RESULTS:\nroc-auc_score:\n',
          #'train:', roc_auc_score(ytrain, yprobtrain, multi_class='ovr'),
          #'test:', roc_auc_score(ytest, yprobtest, multi_class='ovr'),
          '\nf1_score:\n',
          'train:', f1_score(ytrain, ypredtrain, average='macro'),
          'test:', f1_score(ytest, ypredtest, average='macro'),
          '\nclassification_report\ntrain:\n',
          classification_report(ytrain, ypredtrain),
          '\ntest:\n',
          classification_report(ytest, ypredtest)
         )

In [27]:
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2, stratify=y, random_state=12345)

In [28]:
ytrain = ytrain.squeeze()
ytest = ytest.squeeze()

In [32]:
preprocessor = feature_engineering('tfidf', (1, 1))

clfLR = Pipeline(
    steps=[("preprocessor", preprocessor), ("classifier", LogisticRegression(class_weight='balanced', random_state = 12345))]
)

clfSVC = Pipeline(
    steps=[("preprocessor", preprocessor), ("classifier", SVC(class_weight='balanced', probability=True, random_state = 12345))]
)

In [33]:
modelfit(clfLR)

/usr/local/lib/python3.11/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


RESULTS:
roc-auc_score:
 
f1_score:
 train: 0.8874361041766807 test: 0.6345153424417329 
classification_report
train:
               precision    recall  f1-score   support

           0       0.99      0.92      0.96      6096
           1       0.71      0.98      0.82      1219

    accuracy                           0.93      7315
   macro avg       0.85      0.95      0.89      7315
weighted avg       0.95      0.93      0.93      7315
 
test:
               precision    recall  f1-score   support

           0       0.90      0.80      0.84      1524
           1       0.35      0.54      0.43       305

    accuracy                           0.75      1829
   macro avg       0.62      0.67      0.63      1829
weighted avg       0.81      0.75      0.77      1829



In [34]:
modelfit(clfSVC)

/usr/local/lib/python3.11/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


RESULTS:
roc-auc_score:
 
f1_score:
 train: 0.999262435969813 test: 0.5809653830461664 
classification_report
train:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      6096
           1       1.00      1.00      1.00      1219

    accuracy                           1.00      7315
   macro avg       1.00      1.00      1.00      7315
weighted avg       1.00      1.00      1.00      7315
 
test:
               precision    recall  f1-score   support

           0       0.85      0.98      0.91      1524
           1       0.58      0.16      0.25       305

    accuracy                           0.84      1829
   macro avg       0.71      0.57      0.58      1829
weighted avg       0.81      0.84      0.80      1829



## Обучение

In [14]:
import random

In [15]:
def set_random_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_random_seed(12345)

In [16]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred.predictions, eval_pred.label_ids
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro')
    }

# Загружаем обогащённый датасет
df = pd.read_pickle('enriched_sarcasm_pooled.pkl')
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['sarcasm'], random_state=42)

tokenizer = BertTokenizer.from_pretrained('DeepPavlov/rubert-base-cased')
train_ds = SarcasmDataset(train_df, tokenizer)
eval_ds  = SarcasmDataset(test_df,  tokenizer)

bert = BertModel.from_pretrained('DeepPavlov/rubert-base-cased')

model = SarcasmClassifier(
    bert_model=bert,
    sem_dim=len(df['sem_pooled'].iloc[0]),
    num_classes=2
)

training_args = TrainingArguments(
    output_dir="./results",
    do_train=True,
    do_eval=True,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=True,
    dataloader_num_workers=2,

    # частота в шагах
    logging_steps=50,
    eval_steps=500,
    save_steps=500,
    save_total_limit=1,

    load_best_model_at_end=False,
    metric_for_best_model="accuracy",
    report_to="tensorboard",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    compute_metrics=compute_metrics
)

# Тренировка и оценка
trainer.train()
metrics = trainer.evaluate()
print(metrics)

Some weights of the model checkpoint at DeepPavlov/rubert-base-cased were not used when initializing BertModel: ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Step,Training Loss
50,0.487000
100,0.432300
150,0.426700
200,0.388700
250,0.344000
300,0.300200
350,0.330800
400,0.292600
450,0.294100
500,0.235300


{'eval_loss': 0.43770262598991394, 'eval_accuracy': 0.8452706396938218, 'eval_f1_macro': 0.6967456253980306, 'eval_runtime': 3.5631, 'eval_samples_per_second': 513.315, 'eval_steps_per_second': 8.139, 'epoch': 3.0}


In [17]:
'''import torch
import torch.nn as nn
from transformers import BertModel, BertConfig

class CustomBertClassifier(nn.Module):
    def __init__(self,
                 pretrained_model_name: str = 'bert-base-uncased',
                 num_labels: int = 2,
                 hidden_dim: int = 768,
                 dropout_prob: float = 0.1):
        super().__init__()
        # Базовая модель BERT без головы для маскированного языка
        self.bert = BertModel.from_pretrained(pretrained_model_name)

        # Кастомная голова
        self.classifier = nn.Sequential(
            nn.Dropout(dropout_prob),
            nn.Linear(self.bert.config.hidden_size, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
            nn.Linear(hidden_dim, num_labels)
        )

    def forward(self,
                input_ids: torch.LongTensor,
                attention_mask: torch.Tensor = None,
                token_type_ids: torch.Tensor = None,
                labels: torch.LongTensor = None):
        # Получаем выходы из берта
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            return_dict=True
        )
        # Выбираем pooled_output
        pooled_output = outputs.pooler_output

        # Передаём через свою голову
        logits = self.classifier(pooled_output)

        # Если есть метки - считаем loss
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))
            return {
                'loss': loss,
                'logits': logits
            }
        return {'logits': logits}'''

"import torch\nimport torch.nn as nn\nfrom transformers import BertModel, BertConfig\n\nclass CustomBertClassifier(nn.Module):\n    def __init__(self,\n                 pretrained_model_name: str = 'bert-base-uncased',\n                 num_labels: int = 2,\n                 hidden_dim: int = 768,\n                 dropout_prob: float = 0.1):\n        super().__init__()\n        # Базовая модель BERT без головы для маскированного языка\n        self.bert = BertModel.from_pretrained(pretrained_model_name)\n\n        # Кастомная голова\n        self.classifier = nn.Sequential(\n            nn.Dropout(dropout_prob),\n            nn.Linear(self.bert.config.hidden_size, hidden_dim),\n            nn.ReLU(),\n            nn.Dropout(dropout_prob),\n            nn.Linear(hidden_dim, num_labels)\n        )\n\n    def forward(self,\n                input_ids: torch.LongTensor,\n                attention_mask: torch.Tensor = None,\n                token_type_ids: torch.Tensor = None,\n            

In [18]:
'''import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset as TorchDataset, DataLoader
from transformers import BertTokenizer, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from dataclasses import dataclass

combined_df = abc
combined_df['sarcasm'] = combined_df['sarcasm'].astype(int)

train_df, test_df = train_test_split(
    combined_df,
    test_size=0.2,
    stratify=combined_df['sarcasm'],
    random_state=42
)

# Токенизация
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

@dataclass
class NERFeatures:
    input_ids: torch.Tensor
    attention_mask: torch.Tensor
    token_type_ids: torch.Tensor
    labels: torch.Tensor

class SarcasmDataset(TorchDataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.texts = df['text'].tolist()
        self.labels = df['sarcasm'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        enc = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'token_type_ids': enc.get('token_type_ids', torch.zeros_like(enc['input_ids'])).squeeze(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Создаем модели
train_dataset = SarcasmDataset(train_df, tokenizer)
test_dataset  = SarcasmDataset(test_df, tokenizer)

# Загрузка модели
model = CustomBertClassifier(
    pretrained_model_name='bert-base-uncased',
    num_labels=2,
    hidden_dim=768,
    dropout_prob=0.1
)

# Метрики

def compute_metrics(eval_pred):
    logits, labels = eval_pred.predictions, eval_pred.label_ids
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='macro')
    }

# Параметры
training_args = TrainingArguments(
    output_dir="./results",
    do_train=True,
    do_eval=True,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=True,
    dataloader_num_workers=2,

    # частота в шагах
    logging_steps=50,
    eval_steps=500,
    save_steps=500,
    save_total_limit=1,

    load_best_model_at_end=False,
    metric_for_best_model="accuracy",
    report_to="tensorboard",
)

# Трейнер
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()
results = trainer.evaluate()
print(results)
# Save the best model
trainer.save_model('./best_model')'''


'import os\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.utils.data import Dataset as TorchDataset, DataLoader\nfrom transformers import BertTokenizer, Trainer, TrainingArguments\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.metrics import accuracy_score, f1_score\nfrom dataclasses import dataclass\n\ncombined_df = abc\ncombined_df[\'sarcasm\'] = combined_df[\'sarcasm\'].astype(int)\n\ntrain_df, test_df = train_test_split(\n    combined_df,\n    test_size=0.2,\n    stratify=combined_df[\'sarcasm\'],\n    random_state=42\n)\n\n# Токенизация\ntokenizer = BertTokenizer.from_pretrained(\'bert-base-uncased\')\n\n@dataclass\nclass NERFeatures:\n    input_ids: torch.Tensor\n    attention_mask: torch.Tensor\n    token_type_ids: torch.Tensor\n    labels: torch.Tensor\n\nclass SarcasmDataset(TorchDataset):\n    def __init__(self, df, tokenizer, max_length=128):\n        self.texts = df[\'text\'].tolist()\n        self.labels = df[\'sarcasm\'].tolist()\n